# PrintVision AI - retrain and compare (Colab)

Retrains the YOLOv8 defect detector and compares it with the current model on the **same test split**.
The new weights are saved as `best_v2.pt` and are only suggested as a replacement if they beat the current `best.pt`.

**Before running:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Colab left sidebar -> **Secrets (key icon)** -> add `ROBOFLOW_API_KEY` with your key and enable notebook access.
3. Make sure the current model is at `MyDrive/PrintVision_Final/best.pt`.

In [ ]:
%pip install -q ultralytics roboflow
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# ---- settings -------------------------------------------------------------
from pathlib import Path

WORKSPACE, PROJECT, VERSION = 'abilesh-athimulam', 'my-first-project-krsyz', 5
FINAL_DIR   = Path('/content/drive/MyDrive/PrintVision_Final')
OLD_WEIGHTS = FINAL_DIR / 'best.pt'
RUNS_DIR    = Path('/content/drive/MyDrive/PrintVision_Training')

MODEL   = 'yolov8n.pt'   # try 'yolov8s.pt' for more accuracy (slower, ~22 MB)
EPOCHS  = 100            # early stopping (PATIENCE) ends it sooner if it stops improving
PATIENCE = 25
IMGSZ   = 640            # same size the app uses for inference
BATCH   = 16

In [ ]:
# ---- dataset (key comes from Colab Secrets, never typed into the code) ----
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')
DATA_YAML = Path(dataset.location) / 'data.yaml'
print(DATA_YAML.read_text())

In [ ]:
# ---- train ----------------------------------------------------------------
from ultralytics import YOLO

run = YOLO(MODEL).train(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    cos_lr=True,          # smoother learning-rate decay
    close_mosaic=10,      # last 10 epochs without mosaic -> better on real single photos
    degrees=5, fliplr=0.5, hsv_v=0.4,   # mild rotation / flip / brightness (lighting varies)
    seed=0, deterministic=True,
    project=str(RUNS_DIR), name='retrain_v2', exist_ok=False,
)
NEW_WEIGHTS = Path(run.save_dir) / 'weights' / 'best.pt'
print('New weights:', NEW_WEIGHTS)

In [ ]:
# ---- compare old vs new on the same split ---------------------------------
import numpy as np, yaml

split = 'test' if 'test' in yaml.safe_load(DATA_YAML.read_text()) else 'val'

def evaluate(weights):
    m = YOLO(str(weights)).val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, plots=False, verbose=False)
    f1 = m.box.f1_curve.mean(0)          # mean F1 over classes at each confidence
    i = int(np.argmax(f1))
    return dict(F1=float(f1[i]), conf=float(m.box.px[i]), mAP50=float(m.box.map50),
                mAP50_95=float(m.box.map), P=float(m.box.mp), R=float(m.box.mr))

results = {}
if OLD_WEIGHTS.exists():
    results['current best.pt'] = evaluate(OLD_WEIGHTS)
results['retrained'] = evaluate(NEW_WEIGHTS)

import pandas as pd
table = pd.DataFrame(results).T.round(3)
print(f'Evaluated on the {split} split')
table

In [ ]:
# ---- save (never overwrites best.pt) -------------------------------------
import shutil, json
shutil.copy2(NEW_WEIGHTS, FINAL_DIR / 'best_v2.pt')
(FINAL_DIR / 'retrain_v2_metrics.json').write_text(json.dumps(results, indent=2))

new, old = results['retrained'], results.get('current best.pt')
if old is None or new['F1'] > old['F1']:
    print(f"Retrained model is better (F1 {new['F1']:.3f} at conf {new['conf']:.3f}).")
    print('To use it: rename best_v2.pt to best.pt yourself, and set the app confidence to', round(new['conf'], 2))
else:
    print(f"Keep the current model (F1 {old['F1']:.3f} vs retrained {new['F1']:.3f}).")